In [0]:
import pandas as pd
from pyspark.sql import functions as F

In [0]:
master_df = spark.table("prd_mega.sgpbpi163.0a_goat_master")
dli_dlr_data = spark.read.format('csv').option("delimiter",",").option("header","True").load('/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/dli_dlr_goat.csv')
dli_dlr = dli_dlr_data.toPandas()

In [0]:
final_dli_dlr = dli_dlr.groupby("PROJ_ID", as_index=False).agg({
    "DLI": lambda x: " ".join(x.astype(str)),
    "DLR": lambda x: " ".join(x.astype(str))
})

# Display the result
final_dli_dlr

In [0]:
"""
Process DLI data:
- Input columns used: proj_id, proj_apprvl_fy, ind_name, ind_amt
- Output columns: Project ID, Year, DLI

Deduplication rule:
- Within a given (Project ID, Year) group, if two rows have the SAME
  ind_name AND the SAME ind_amt, they are considered duplicates -> keep only one.
- Rows with same ind_name but DIFFERENT ind_amt are NOT deduplicated (treated individually).
- Rows with different ind_name are always kept, even if ind_amt happens to match.

Final DLI string per (Project ID, Year):
  "B[p], D[r]"  -> each deduplicated (ind_name, ind_amt) pair formatted as "name[amt]",
                    joined by ", "
"""

import pandas as pd

INPUT_PATH = ""
OUTPUT_XLSX_PATH = ""
OUTPUT_CSV_PATH = ""

# ---- Load data ----
df = pd.read_excel(INPUT_PATH, sheet_name="DLIs OPDC MEGA(result)")

# ---- Select & rename relevant columns ----
cols_map = {
    "proj_id": "Project ID",
    "proj_apprvl_fy": "Year",
    "ind_name": "DLI_Name",
    "ind_amt": "DLI_Amt",
}
data = df[list(cols_map.keys())].rename(columns=cols_map)

# ---- Clean up: strip whitespace on text fields, keep amounts as-is ----
data["DLI_Name"] = data["DLI_Name"].apply(
    lambda x: "" if pd.isna(x) else str(x).strip()
)

# ---- Step 1: Deduplicate rows where (Project ID, Year, DLI_Name, DLI_Amt) repeat ----
# This keeps a row unique per DLI-amount pair within a project/year;
# same-name-different-amount rows are preserved individually;
# different-name-same-amount rows are also preserved individually.
dedup = data.drop_duplicates(subset=["Project ID", "Year", "DLI_Name", "DLI_Amt"]).copy()


# ---- Step 2: Format each row as "Name[$Amt]" (or just "Name" if amount is 0/blank) ----
def format_amt(amt):
    # Format numeric amounts cleanly (avoid trailing .0 for whole numbers)
    if pd.isna(amt):
        return ""
    if isinstance(amt, (int,)) or (isinstance(amt, float) and float(amt).is_integer()):
        return str(int(amt))
    return str(amt)


def format_name(name):
    return "" if pd.isna(name) else str(name)


def format_row(row):
    name = format_name(row["DLI_Name"])
    amt = row["DLI_Amt"]
    amt_str = format_amt(amt)
    # No brackets when amount is 0, missing, or blank
    if pd.isna(amt) or amt_str == "" or amt_str == "0":
        return name
    return f"{name} [${amt_str}]"


dedup["DLI_Formatted"] = dedup.apply(format_row, axis=1)

# De-dupe again on the final formatted text itself, since a 0-amount and a
# missing-amount row for the same DLI name both collapse to a bare "Name"
# (no brackets), which would otherwise show up twice.
dedup = dedup.drop_duplicates(subset=["Project ID", "Year", "DLI_Formatted"])

# ---- Step 3: Group by Project ID & Year, combine into single string ----
result = (
    dedup.groupby(["Project ID", "Year"])["DLI_Formatted"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
    .rename(columns={"DLI_Formatted": "DLI"})
)

# ---- Save output ----
# NOTE: A small number of Project/Year combinations have a very large number of
# deduplicated DLI[amount] entries, and the combined string can exceed Excel's
# hard 32,767-character-per-cell limit (Excel will silently truncate such cells).
# CSV has no such limit, so CSV is the authoritative/complete output;
# the xlsx copy is provided for convenience but may truncate a few long cells.
import os
os.makedirs("/mnt/user-data/outputs", exist_ok=True)
result.to_csv(OUTPUT_CSV_PATH, index=False)
result.to_excel(OUTPUT_XLSX_PATH, index=False)

long_cells = result[result["DLI"].str.len() > 32767]
print(f"Done. Rows in: {len(data)}, rows after dedup: {len(dedup)}, final project-year rows: {len(result)}")
if len(long_cells):
    print(f"WARNING: {len(long_cells)} row(s) exceed Excel's 32767-char cell limit and will be truncated in the .xlsx (full data is intact in the .csv):")
    print(long_cells[["Project ID", "Year"]].to_string(index=False))
print(f"Saved to: {OUTPUT_CSV_PATH}")
print(f"Saved to: {OUTPUT_XLSX_PATH}")

In [0]:
final_dli_dlr.shape

In [0]:
master_df_pd = master_df.toPandas()
df_result_cleaned = master_df_pd.merge(
    final_dli_dlr,
    on="PROJ_ID",
    how="left"
)

In [0]:
# %sql
# DELETE FROM `prd_mega`.`sgpbpi163`.`0c_hierarchy_table_goat`
# WHERE Valid_Hierarchy = 'False';

In [0]:
from pyspark.sql import functions as F
def generate_overall_goat_df(goat_master_df, hierarchy_df):
    """
    Groups keywords by hierarchy, evaluates their presence across specific text 
    columns in the master dataframe, and produces a long-form DataFrame.
    """
    
    # 1. Aggregate keywords into an array and a regex pattern per hierarchy
    # Note: Based on your image visualization, the column name is 'hierarchy'
    hierarchy_grouped = hierarchy_df \
        .groupBy("hierarchy") \
        .agg(F.collect_list(F.lower(F.col("keyword"))).alias("keyword_list")) \
        .withColumn("regex_pattern", F.concat(F.lit("(?i)\\b("), F.array_join("keyword_list", "|"), F.lit(")\\b")))

    # 2. Extract the search columns and define missing column safety checks
    search_columns = ['Indicators', 'PriorActions', 'PROJ_DEV_OBJECTIVE_DESC', 'Components', 'DLI', 'DLR']
    missing_cols = [col for col in search_columns if col not in goat_master_df.columns]
    if missing_cols:
        raise ValueError(f"Columns missing from goat_master_df: {missing_cols}")

    # 3. Concatenate the text across target columns to search within them all at once
    # Creates a temporary combined text column for evaluation
    goat_master_df = goat_master_df.withColumn(
        "_combined_text", 
        F.concat_ws(" ", *[F.coalesce(F.col(col), F.lit("")) for col in search_columns])
    )
    
    # 4. Cross join the master data with the aggregated hierarchies
    crossed_df = goat_master_df.crossJoin(F.broadcast(hierarchy_grouped))

    # 5. Native SQL evaluation allows evaluating column-against-column regex matches
    overall_goat_df = crossed_df \
        .withColumn(
            "Ishierarchy_present", 
            F.when(F.expr("_combined_text RLIKE regex_pattern"), "Yes").otherwise("No")
        ) \
        .drop("keyword_list", "regex_pattern", "_combined_text") \
        .withColumnRenamed("hierarchy", "hierarchy_name")

    return overall_goat_df

In [0]:
hierarchy_table = spark.table("prd_mega.sgpbpi163.0c_hierarchy_table_goat")

In [0]:
# spark_df = spark.createDataFrame(df_result_cleaned)
# # Write the Spark DataFrame as a Delta table in the target schema
# spark_df.write.format("delta").mode("overwrite").saveAsTable("prd_mega.sgpbpi163.1a_goat_master")

goat_master = spark.table("prd_mega.sgpbpi163.1a_goat_master")

In [0]:
final_data = generate_overall_goat_df(goat_master_df = goat_master, hierarchy_df= hierarchy_table)

In [0]:
display(final_data)

In [0]:
final_data.write.format("delta").mode("overwrite").saveAsTable("prd_mega.sgpbpi163.1b_overall_goat_df")

In [0]:
%sql
-- Step 1: Add the new column to the table schema
-- ALTER TABLE prd_mega.sgpbpi163.1b_overall_goat_df 
-- ADD COLUMN Valid_Hierarchy STRING;

-- Step 2: Populate all rows in the new column with "Yes"
UPDATE prd_mega.sgpbpi163.1b_overall_goat_df 
SET Valid_Hierarchy = 'True';

With DLI Mater File

In [0]:
from pyspark.sql import functions as F

# Load the CSV into a Spark DataFrame
dli_master_data = (
    spark.read.format('csv')
    # .option("delimiter", ",")
    .option("header", "True")
    .load('/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/DLI/DLIs OPDC MEGA.csv')
)

dli_master = dli_master_data.filter(
    F.col("PROJ_ID") != '- Standard regulation for prov"'
)
# Use Spark DataFrame aggregation to concatenate ind_name per proj_id
final_dli_master = (
    dli_master
    .groupBy("PROJ_ID")
    .agg(F.concat_ws(" ", F.collect_list(F.col("DLI"))).alias("DLI"))
)

# Display the result
final_dli_master

In [0]:
dli_master_data = pd.read_excel("/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/DLI/DLIs OPDC MEGA_with_year.xlsx")
final_dli_master = dli_master_data.groupby("proj_id", as_index=False).agg({
    "ind_name": lambda x: " ".join(x.astype(str)),
    "proj_apprvl_fy": "first"
})

In [0]:
# Rename columns if needed before displaying
final_dli_master = (
    final_dli_master
    .rename(columns={"proj_id": "PROJ_ID"})
    .rename(columns={"ind_name": "DLI"})
    .rename(columns={"proj_apprvl_fy":"FY"})
)

display(final_dli_master)

In [0]:
# Count total unique project IDs in the final_data DataFrame
# Filter to FY <= 2026 if the FY column exists
filtered_master = final_dli_master[(final_dli_master["FY"] > 2000) & (final_dli_master["FY"] <= 2026)]

# Count total unique project IDs in the filtered DataFrame
unique_proj_count = filtered_master["PROJ_ID"].nunique()

# Create a one‑row Spark DataFrame to display the result
result_df = spark.createDataFrame(
    [(unique_proj_count,)], 
    ["unique_project_id_count"]
)

display(result_df)

In [0]:
# Create a sample DataFrame containing only the row(s) with proj_id = 'P222518'
sample_df = final_dli_master.filter("PROJ_ID = 'P509827'")

# Display the sample DataFrame
display(sample_df)

In [0]:
test_dli_data = spark.read.format('csv').option("delimiter",",").option("header","True").load('/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/dli_dlr_goat.csv')
test_dli = test_dli_data.toPandas()
test_dli_file = test_dli.groupby("PROJ_ID", as_index=False).agg({
    "DLI": lambda x: " ".join(x.astype(str)),
    # "DLR": lambda x: " ".join(x.astype(str))
})

# Display the result
test_dli_file

In [0]:
import pandas as pd
fin_data = final_dli_master.toPandas()
df_result = pd.DataFrame(test_dli_file).merge(
    fin_data,
    on="PROJ_ID",
    how="left"
)

In [0]:
display(df_result)

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

# 1. Read the Excel file into a Pandas DataFrame using openpyxl
pdf = pd.read_excel(
    '/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/DLI/L9.xlsx', 
    sheet_name=0  # Reads the first sheet by default
)

In [0]:
pdf = pdf[(pdf["Lending Instr. Type"] == "PforR") & (pdf["FY"] <= 2026)]

In [0]:
# Rename the column in pandas DataFrames
if "Project ID" in pdf.columns:
    pdf.rename(columns={"Project ID": "PROJ_ID"}, inplace=True)

# Rename the column in Spark DataFrames (if present)
# if "Project ID" in goat_master.columns:
#     goat_master = goat_master.withColumnRenamed("Project ID", "PROJ_ID")
# if "Project ID" in hierarchy_table.columns:
#     hierarchy_table = hierarchy_table.withColumnRenamed("Project ID", "PROJ_ID")

In [0]:
df_result = pd.DataFrame(pdf).merge(
    fin_data,
    on="PROJ_ID",
    how="left"
)

In [0]:
df_result

In [0]:
df_result_ = df_result[['PROJ_ID', 'Project Name', 'FY','DLI']]

In [0]:
df_result_ = df_result_.drop_duplicates(subset=["PROJ_ID"])
display(df_result_)

In [0]:
# DataFrame with projects where DLI is missing
df_dli_nan = df_result_[df_result_['DLI'].isna()]

display(df_dli_nan) #P167598

In [0]:
df_result_.nunique()

In [0]:
df_result_

In [0]:
# Assuming the pandas DataFrames `df_result_` and `filtered_master` already exist in the notebook

# -------------------------------------------------
# 1. Projects in df_result_ but NOT in filtered_master
# -------------------------------------------------
df_opcs_not_in_master = (
    df_result_[~df_result_["PROJ_ID"].isin(filtered_master["PROJ_ID"])]
    [["PROJ_ID", "FY"]]
)

# -------------------------------------------------
# 2. Projects in filtered_master but NOT in df_result_
# -------------------------------------------------
df_master_not_in_opcs = (
    filtered_master[~filtered_master["PROJ_ID"].isin(df_result_["PROJ_ID"])]
    [["PROJ_ID", "FY"]]
)

# Display the two anti‑join results
display(df_opcs_not_in_master)
display(df_master_not_in_opcs)

# -------------------------------------------------
# 3. Match‑matrix: all projects with presence flags
# -------------------------------------------------
# Union of unique PROJ_ID/FY pairs from both sources
all_projects = (
    pd.concat([df_result_[["PROJ_ID", "FY"]], filtered_master[["PROJ_ID", "FY"]]])
    .drop_duplicates()
    .reset_index(drop=True)
)

# Add Yes/No flags for each source
all_projects["OPCS"] = (
    all_projects["PROJ_ID"]
    .isin(df_result_["PROJ_ID"])
    .map({True: "Yes", False: "No"})
)

all_projects["MEGA"] = (
    all_projects["PROJ_ID"]
    .isin(filtered_master["PROJ_ID"])
    .map({True: "Yes", False: "No"})
)

# Display the match matrix
display(all_projects)

In [0]:
# Filter projects where DLI Master = Yes and OPCS = No
projects_missing_opcs = all_projects[
    (all_projects["DLI Master"] == "Yes") & (all_projects["OPCS"] == "No")
]

display(projects_missing_opcs)

In [0]:
# # Convert the pandas DataFrame `fin` to a Spark DataFrame and save as a Delta table
# spark_fin_df = spark.createDataFrame(fin_data)

# spark_fin_df.write.format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("prd_mega.sgpbpi163.0_goat_dli_master")

In [0]:
# Load the Delta table as a Spark DataFrame
goat_master_df = spark.table("prd_mega.sgpbpi163.`0a_goat_master`")

# Collect distinct PROJ_IDs from the master table
master_proj_ids = set(
    goat_master_df.select("PROJ_ID")
                 .distinct()
                 .rdd.map(lambda r: r[0])
                 .collect()
)

# Identify PROJ_IDs from the pandas result that are missing in the master table
missing_proj_ids = [
    pid for pid in df_result_["PROJ_ID"]
    if pid not in master_proj_ids
]

# Show whether all PROJ_IDs are present and list any missing ones
if missing_proj_ids:
    display(
        spark.createDataFrame(
            pd.DataFrame({"Missing_PROJ_ID": missing_proj_ids})
        )
    )
else:
    display(spark.createDataFrame(pd.DataFrame({"All_PROJ_IDs_present": [True]})))

In [0]:
datapdf = pd.read_excel(
    '/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/DLI/L9.xlsx', 
    sheet_name=0  # Reads the first sheet by default
)

In [0]:
datapdf

In [0]:
# Ensure the Excel dataframe has a PROJ_ID column
if "Project ID" in datapdf.columns:
    datapdf = datapdf.rename(columns={"Project ID": "PROJ_ID"})

# Pandas dataframe of missing project IDs
missing_pd = pd.DataFrame({"Missing_PROJ_ID": missing_proj_ids})

# Merge FY information from the Excel data
missing_with_fy = (
    missing_pd.merge(
        datapdf[["PROJ_ID", "FY", "Project Name"]],
        left_on="Missing_PROJ_ID",
        right_on="PROJ_ID",
        how="left",
    )
    .drop(columns="PROJ_ID")
)

# Show as a Spark dataframe
display(spark.createDataFrame(missing_with_fy))

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

# 1. Load the GOAT master Delta table into a pandas DataFrame
goat_df = spark.table("prd_mega.sgpbpi163.2a_goat_master").toPandas()

# 2. Load the Mega DLI Excel file into a pandas DataFrame
# mega_dlis = pd.read_excel("/Volume/WorldBank/vgpbpi163/GOAT/master.xlsx")
mega_dlis_1 = pd.read_excel("/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/DLI/DLIs OPDC MEGA_with_year.xlsx")
mega_dlis = mega_dlis_1.groupby("proj_id", as_index=False).agg({
    "proj_id": "first",
    "ind_name": lambda x: " ".join(x.astype(str)),
    "proj_apprvl_fy": "first"
}).rename(columns={"proj_id": "PROJ_ID"})

# Optional: standardize column names if they differ
if "Project ID" in mega_dlis.columns:
    mega_dlis = mega_dlis.rename(columns={"Project ID": "PROJ_ID"})
if "Project Name" in mega_dlis.columns:
    mega_dlis = mega_dlis.rename(columns={"Project Name": "PROJECT_NAME"})
if "proj_apprvl_fy" in mega_dlis.columns:
    mega_dlis = mega_dlis.rename(columns={"proj_apprvl_fy": "FY"})

# 3. Identify projects present in mega_dlis but NOT in goat, with FY <= 2026
missing_projects = mega_dlis[
    (~mega_dlis["PROJ_ID"].isin(goat_df["PROJ_ID"])) &
    (mega_dlis["FY"] <= 2026)
][["PROJ_ID", "FY"]]

# 4. Display the result
display(missing_projects)

In [0]:
pforr_projects = mega_dlis[
    (mega_dlis["FY"] <= 2026)
][["PROJ_ID", "FY"]]
display(pforr_projects)

In [0]:
# 3. Identify projects present in goat and NOT in mega_dlis, with FY <= 2026
missing_projects_2 = goat_df[
    (~goat_df["PROJ_ID"].isin(mega_dlis["PROJ_ID"])) &
    (goat_df["PROJ_APPRVL_FY"].astype(int) <= 2026) &
    (goat_df["LNDNG_INSTR_LONG_NAME"] == "Program-for-Results Financing")
][["PROJ_ID", "PROJ_APPRVL_FY"]]

display(missing_projects_2)

In [0]:
goat_df